In [4]:
import json
from collections import Counter

# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_PATH  = "cjpe_train_rr_segmented.jsonl"
OUTPUT_PATH = "cjpe_track_B_extracted.jsonl"

# ── Track B fields: Ratio-Grounded (RATIO + STATUTE + ISSUE) ─────────────────
# Original key names as they appear in the JSONL
FIELD_MAP = {
    "RATIO"   : "RATIO",    # Ratio of the Decision
    "STATUTE" : "STA",      # Statute  (key in JSONL is STA)
    "ISSUE"   : "ISSUE",    # Issue
}

# ── Counters ──────────────────────────────────────────────────────────────────
extracted      = 0
empty_ratio    = 0
empty_statute  = 0
empty_issue    = 0

# ── Extraction ────────────────────────────────────────────────────────────────
with open(INPUT_PATH, "r", encoding="utf-8") as fin, \
     open(OUTPUT_PATH, "w", encoding="utf-8") as fout:

    for line in fin:
        line = line.strip()
        if not line:
            continue

        rec = json.loads(line)

        out = {
            "id"      : rec.get("id", ""),
            "RATIO"   : rec.get("RATIO", "").strip(),
            "STATUTE" : rec.get("STA",   "").strip(),   # STA is the JSONL key
            "ISSUE"   : rec.get("ISSUE", "").strip(),
            "label"   : rec.get("label", -1),
        }

        # ── Track empty fields ────────────────────────────────────────────────
        if not out["RATIO"]:   empty_ratio   += 1
        if not out["STATUTE"]: empty_statute += 1
        if not out["ISSUE"]:   empty_issue   += 1

        fout.write(json.dumps(out, ensure_ascii=False) + "\n")
        extracted += 1

# ═════════════════════════════════════════════════════════════════════════════
# Summary
# ═════════════════════════════════════════════════════════════════════════════
print("=" * 55)
print("  TRACK B — RATIO-GROUNDED EXTRACTION COMPLETE")
print("=" * 55)
print(f"  Input          : {INPUT_PATH}")
print(f"  Output         : {OUTPUT_PATH}")
print(f"  Total records  : {extracted:,}")
print()
print(f"  Field coverage (docs that HAVE the field):")
print(f"    RATIO   : {extracted - empty_ratio:>6,}  /  {extracted:,}  "
      f"({(extracted - empty_ratio)   / extracted * 100:.1f}%)")
print(f"    STATUTE : {extracted - empty_statute:>6,}  /  {extracted:,}  "
      f"({(extracted - empty_statute) / extracted * 100:.1f}%)")
print(f"    ISSUE   : {extracted - empty_issue:>6,}  /  {extracted:,}  "
      f"({(extracted - empty_issue)   / extracted * 100:.1f}%)")

# ── Label distribution ────────────────────────────────────────────────────────
label_counts = Counter()
with open(OUTPUT_PATH) as f:
    for line in f:
        label_counts[json.loads(line)["label"]] += 1

print()
print(f"  Label distribution:")
for lbl, cnt in sorted(label_counts.items()):
    name = "ACCEPTED" if lbl == 1 else "REJECTED"
    print(f"    Label {lbl} ({name}) : {cnt:>6,}  ({cnt / extracted * 100:.1f}%)")

# ── Sample output (first 3 records) ──────────────────────────────────────────
print()
print("  Sample output (first 3 records):")
with open(OUTPUT_PATH) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        rec = json.loads(line)
        print(f"\n  {'─' * 52}")
        print(f"  ID      : {rec['id']}  |  Label: {rec['label']}")
        print(f"  RATIO   : {str(rec['RATIO'])[:120]}"
              f"{'...' if len(str(rec['RATIO']))   > 120 else ''}")
        print(f"  STATUTE : {str(rec['STATUTE'])[:120]}"
              f"{'...' if len(str(rec['STATUTE'])) > 120 else ''}")
        print(f"  ISSUE   : {str(rec['ISSUE'])[:120]}"
              f"{'...' if len(str(rec['ISSUE']))   > 120 else ''}")

print(f"\n  Output saved to → {OUTPUT_PATH}")

  TRACK B — RATIO-GROUNDED EXTRACTION COMPLETE
  Input          : cjpe_train_rr_segmented.jsonl
  Output         : cjpe_track_B_extracted.jsonl
  Total records  : 32,191

  Field coverage (docs that HAVE the field):
    RATIO   : 11,287  /  32,191  (35.1%)
    STATUTE : 11,556  /  32,191  (35.9%)
    ISSUE   :  9,063  /  32,191  (28.2%)

  Label distribution:
    Label 0 (REJECTED) : 18,856  (58.6%)
    Label 1 (ACCEPTED) : 13,335  (41.4%)

  Sample output (first 3 records):

  ────────────────────────────────────────────────────
  ID      : 2020_1  |  Label: 0
  RATIO   : 
  STATUTE : 
  ISSUE   : Uday Umesh Lalit, J. These appeals arise out of the Judgment and Order dated 09.12.2015 passed by the Division Bench of ...

  ────────────────────────────────────────────────────
  ID      : 2020_2  |  Label: 0
  RATIO   : 
  STATUTE : 
  ISSUE   : Indira Banerjee, J. These appeals are against the judgment and order dated 21.11.2006 passed by the Madurai Bench of Mad...

  ─────────────────